In [2]:
# This is for JJA season, for DJF season is at "RFandSHAP_rev3_boostrap.ipynb"

In [8]:
"""
JJA SEGMENT 1 of 8: HEADER, IMPORTS, AND CLASS INITIALIZATION

This is the beginning of the JJA script.
Copy this segment first into your new JJA script file.

Season: JJA (June + July + August from same calendar year)
Data: Dec 2020 to Nov 2050 → JJA seasons 2021-2050 (30 seasons)
"""

# ============================================================================
# ESSENTIAL IMPORTS
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
import networkx as nx
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# SHAP import with installation check
try:
    import shap
    SHAP_AVAILABLE = True
    print("✓ SHAP library available - enhanced interpretability enabled!")
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠ SHAP library not found. Install with: pip install shap")
    print("  SHAP analysis will be skipped, but other features will work normally.")

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# ============================================================================
# CLASS DEFINITION AND INITIALIZATION
# ============================================================================

class ClimateMutualFeedbackAnalyzer_JJA_SHAP:
    """
    Enhanced Climate Mutual Feedback Analysis System for JJA Seasons
    
    Features:
    - JJA seasonal analysis (June + July + August from same year)
    - Random Forest feature importance
    - Granger causality testing
    - SHAP interpretability analysis
    - Block bootstrap uncertainty quantification
    - Normalized feedback strengths
    - Multi-method validation
    
    Data period: Dec 2020 - Nov 2050 → JJA seasons 2021-2050 (30 seasons)
    """
    
    def __init__(self, max_lags=6, significance_level=0.05, n_estimators=200, 
                 random_state=42, n_bootstrap=0, block_length=None):
        """
        Initialize the JJA Climate Mutual Feedback Analyzer
        
        Parameters:
        -----------
        max_lags : int
            Maximum number of lags for Granger causality testing
        significance_level : float
            Significance level for statistical tests (default: 0.05)
        n_estimators : int
            Number of trees in Random Forest (default: 200)
        random_state : int
            Random seed for reproducibility
        n_bootstrap : int
            Number of bootstrap samples (0 to disable, 1000 recommended)
        block_length : int or None
            Block length for block bootstrap (auto-calculated if None)
        """
        self.max_lags = max_lags
        self.significance_level = significance_level
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.n_bootstrap = n_bootstrap  # Set to 0 to disable, 1000 to enable
        self.block_length = block_length  # Auto-calculated if None
        
        # Enhanced variable mapping for multiple datasets
        self.climate_variables = {
            'temperature': {
                'cmip6': ['tas', 'ta'],
                'era5': ['t2m', '2t'],
                'generic': ['temp', 'temperature']
            },
            'precipitable_water': {
                'cmip6': ['prw'],
                'era5': ['tcwv'],
                'generic': ['pwat', 'precipitable_water']
            },
            'precipitation': {
                'cmip6': ['pr'],
                'era5': ['tp', 'mtpr'],
                'generic': ['precip', 'precipitation']
            },
            'evaporation': {
                'cmip6': ['hfls', 'evspsbl'],
                'era5': ['e', 'slhf'],
                'generic': ['evap', 'evaporation']
            },
            'soil_moisture': {
                'cmip6': ['mrsos', 'mrlsl'],
                'era5': ['swvl1', 'swvl2', 'swvl3', 'swvl4'],
                'generic': ['sm', 'soil_moisture']
            },
            'runoff': {
                'cmip6': ['mrro', 'mrros'],
                'era5': ['ro'],
                'generic': ['runoff', 'qs']
            }
        }
        
        # Unit conversion information
        self.unit_conversions = {
            'precipitation': {
                'pr': {'factor': 86400, 'description': 'kg/m²/s to mm/day'},
                'tp': {'factor': 1000, 'description': 'm/day to mm/day'},
                'mtpr': {'factor': 86400, 'description': 'kg/m²/s to mm/day'},
                'precip': {'factor': 1, 'description': 'assumed already in mm/day'}
            },
            'evaporation': {
                'hfls': {'factor': 86400 / (2.45 * 1e6), 'description': 'W/m² to mm/day (latent heat)'},
                'evspsbl': {'factor': 86400, 'description': 'kg/m²/s to mm/day'},
                'e': {'factor': 1000, 'description': 'm/day to mm/day'},
                'slhf': {'factor': 86400 / (2.45 * 1e6), 'description': 'W/m² to mm/day (latent heat)'},
                'evap': {'factor': 1, 'description': 'assumed already in mm/day'}
            },
            'soil_moisture': {
                'mrsos': {'factor': 1 / (1000 * 0.1), 'description': 'kg/m² to m (10cm layer)'},
                'mrlsl': {'factor': 1, 'description': 'assumed already in m³/m³'},
                'swvl1': {'factor': 1, 'description': 'already in m³/m³'},
                'swvl2': {'factor': 1, 'description': 'already in m³/m³'},
                'swvl3': {'factor': 1, 'description': 'already in m³/m³'},
                'swvl4': {'factor': 1, 'description': 'already in m³/m³'},
                'sm': {'factor': 1, 'description': 'assumed already in m³/m³'}
            },
            'runoff': {
                'mrro': {'factor': 86400, 'description': 'kg/m²/s to mm/day'},
                'mrros': {'factor': 86400, 'description': 'kg/m²/s to mm/day'},
                'ro': {'factor': 1000, 'description': 'm/day to mm/day'},
                'runoff': {'factor': 1, 'description': 'assumed already in mm/day'},
                'qs': {'factor': 1, 'description': 'assumed already in mm/day'}
            },
            'temperature': {
                'tas': {'factor': 1, 'description': 'K (will convert to °C)'},
                'ta': {'factor': 1, 'description': 'K (will convert to °C)'},
                't2m': {'factor': 1, 'description': 'K (will convert to °C)'},
                '2t': {'factor': 1, 'description': 'K (will convert to °C)'},
                'temp': {'factor': 1, 'description': 'assumed already in °C'}
            },
            'precipitable_water': {
                'prw': {'factor': 1, 'description': 'kg/m² (no conversion needed)'},
                'tcwv': {'factor': 1, 'description': 'kg/m² (no conversion needed)'},
                'pwat': {'factor': 1, 'description': 'kg/m² (no conversion needed)'}
            }
        }
        
        # Regional definitions
        self.regions = {
            "Global": {"lat_min": -60, "lat_max": 60},
            "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
            "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
            "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
            "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
            "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
        }
        
        # Storage dictionaries
        self.datasets = {}  # {dataset_name: {variable: data}}
        self.regional_data = {}  # {dataset_name: {region: DataFrame}}
        self.conversion_log = {}  # {dataset_name: {variable: conversion_info}}
        self.feedback_results = {}  # {dataset_name: {region: feedback_results}}
        self.rf_predictions = {}  # Store RF predictions for evaluation plots
        
        # Print initialization summary
        print("="*70)
        print("Enhanced JJA Climate Mutual Feedback Analyzer Initialized!")
        print("="*70)
        print(f"Supported datasets: CMIP6, ERA5, Generic")
        print(f"Expected variables: {list(self.climate_variables.keys())}")
        print(f"Regions to analyze: {list(self.regions.keys())}")
        print(f"Season: JJA (June + July + August)")
        print(f"Expected data period: Dec 2020 - Nov 2050 → JJA seasons 2021-2050 (30 seasons)")
        print(f"\nAnalysis methods:")
        print(f"  • Random Forest feature importance")
        print(f"  • Granger Causality testing")
        print(f"  • SHAP interpretability ({'ENABLED' if SHAP_AVAILABLE else 'DISABLED'})")
        print(f"  • Block Bootstrap uncertainty ({'ENABLED' if n_bootstrap > 0 else 'DISABLED'})")
        print(f"\nParameters:")
        print(f"  • max_lags: {max_lags} JJA seasons")
        print(f"  • significance_level: {significance_level}")
        print(f"  • n_estimators: {n_estimators}")
        print(f"  • random_state: {random_state}")
        print(f"  • n_bootstrap: {n_bootstrap} samples")
        if n_bootstrap > 0:
            print(f"  • Block bootstrap: ENABLED ({n_bootstrap} samples)")
            print(f"  • Block length: {'auto-calculated' if block_length is None else block_length}")
        print("="*70)


    def detect_dataset_type(self, file_path, variable_name):
        """
        Detect whether file is CMIP6, ERA5, or generic format
        
        Parameters:
        -----------
        file_path : str
            Path to NetCDF file
        variable_name : str
            Expected variable name (e.g., 'temperature', 'precipitation')
            
        Returns:
        --------
        tuple : (dataset_type, variable_code)
            e.g., ('cmip6', 'tas') or ('era5', 't2m')
        """
        try:
            import xarray as xr
            ds = xr.open_dataset(file_path)
            var_names = list(ds.data_vars.keys())
            
            if variable_name in self.climate_variables:
                var_dict = self.climate_variables[variable_name]
                
                # Check for ERA5 first (more specific naming)
                for era5_name in var_dict.get('era5', []):
                    if era5_name in var_names:
                        return 'era5', era5_name
                
                # Check for CMIP6
                for cmip6_name in var_dict.get('cmip6', []):
                    if cmip6_name in var_names:
                        return 'cmip6', cmip6_name
                
                # Check for generic
                for generic_name in var_dict.get('generic', []):
                    if generic_name in var_names:
                        return 'generic', generic_name
            
            # Fallback - exclude auxiliary variables
            excluded_vars = ['time_bnds', 'lat_bnds', 'lon_bnds', 'bnds', 'bounds']
            valid_vars = [v for v in var_names if not any(excl in v.lower() for excl in excluded_vars)]
            
            if valid_vars:
                return 'unknown', valid_vars[0]
            
            return 'unknown', None
            
        except Exception as e:
            print(f"Warning: Could not detect dataset type: {str(e)}")
            return 'unknown', None
    
    def _apply_unit_conversion(self, data_var, variable_name, var_code):
        """
        Apply unit conversions to climate variables
        
        Handles conversions like:
        - Temperature: K to °C
        - Precipitation: kg/m²/s to mm/day
        - Evaporation: W/m² to mm/day (via latent heat)
        - Soil moisture: kg/m² to m³/m³
        - Runoff: kg/m²/s to mm/day
        """
        conversion_info = {
            'original_var_code': var_code,
            'conversion_applied': False,
            'factor': 1.0,
            'description': 'No conversion needed',
            'final_units': data_var.attrs.get('units', 'unknown')
        }
        
        if variable_name in self.unit_conversions:
            var_conversions = self.unit_conversions[variable_name]
            
            if var_code in var_conversions:
                conv_info = var_conversions[var_code]
                factor = conv_info['factor']
                description = conv_info['description']
                
                if factor != 1.0:
                    print(f"  Applying conversion: {description}")
                    converted_data = data_var * factor
                    new_attrs = data_var.attrs.copy()
                    
                    # Set appropriate final units
                    if variable_name == 'precipitation':
                        new_attrs['units'] = 'mm/day'
                    elif variable_name == 'evaporation':
                        new_attrs['units'] = 'mm/day'
                    elif variable_name == 'soil_moisture':
                        new_attrs['units'] = 'm³/m³'
                    elif variable_name == 'runoff':
                        new_attrs['units'] = 'mm/day'
                    elif variable_name == 'temperature' and var_code in ['tas', 'ta', 't2m', '2t']:
                        converted_data = converted_data - 273.15
                        new_attrs['units'] = 'degrees_C'
                        description += ' + K to °C'
                    
                    converted_data.attrs = new_attrs
                    
                    conversion_info.update({
                        'conversion_applied': True,
                        'factor': factor,
                        'description': description,
                        'final_units': new_attrs['units']
                    })
                    
                    return converted_data, conversion_info
                else:
                    # Handle temperature conversion even if factor is 1
                    if variable_name == 'temperature' and var_code in ['tas', 'ta', 't2m', '2t']:
                        sample_values = data_var.values.flatten()
                        sample_values = sample_values[~np.isnan(sample_values)][:1000]
                        
                        if len(sample_values) > 0 and np.mean(sample_values) > 200:
                            print(f"  Converting temperature from K to °C")
                            converted_data = data_var - 273.15
                            new_attrs = data_var.attrs.copy()
                            new_attrs['units'] = 'degrees_C'
                            converted_data.attrs = new_attrs
                            
                            conversion_info.update({
                                'conversion_applied': True,
                                'factor': 1.0,
                                'description': 'K to °C conversion',
                                'final_units': 'degrees_C'
                            })
                            
                            return converted_data, conversion_info
        
        return data_var, conversion_info
    
    def upload_file(self, dataset_name, variable_name, file_path, custom_var_name=None):
        """
        Upload a NetCDF file for JJA mutual feedback analysis
        
        Parameters:
        -----------
        dataset_name : str
            Name for this dataset (e.g., 'cmip6', 'era5')
        variable_name : str
            Expected variable type (e.g., 'temperature', 'precipitation')
        file_path : str
            Path to NetCDF file
        custom_var_name : str, optional
            Override automatic variable detection
            
        Returns:
        --------
        bool : True if successful, False otherwise
        """
        try:
            import xarray as xr
            
            print(f"\nUploading {variable_name} for {dataset_name}")
            print(f"  File: {file_path}")
            
            # Initialize storage
            if dataset_name not in self.datasets:
                self.datasets[dataset_name] = {}
                self.conversion_log[dataset_name] = {}
            
            # Load dataset
            ds = xr.open_dataset(file_path)
            
            # Find the correct variable
            if custom_var_name and custom_var_name in ds.data_vars:
                data_var = ds[custom_var_name]
                var_code = custom_var_name
                detected_type = 'custom'
            else:
                detected_type, var_code = self.detect_dataset_type(file_path, variable_name)
                
                if var_code and var_code in ds.data_vars:
                    data_var = ds[var_code]
                    
                    # Validate spatial dimensions
                    actual_dims = set(data_var.dims)
                    
                    if not any(dim in actual_dims for dim in ['lat', 'latitude']) or \
                       not any(dim in actual_dims for dim in ['lon', 'longitude']):
                        print(f"  Warning: Variable {var_code} doesn't have proper spatial dimensions")
                        var_code = None
                        data_var = None
                    else:
                        print(f"  Detected {detected_type.upper()} dataset with variable '{var_code}'")
                else:
                    var_code = None
                    data_var = None
                
                # If detection failed, try all possible names
                if data_var is None:
                    all_possible_names = []
                    if variable_name in self.climate_variables:
                        for dataset_type, names in self.climate_variables[variable_name].items():
                            all_possible_names.extend(names)
                    
                    for name in all_possible_names:
                        if name in ds.data_vars:
                            candidate_var = ds[name]
                            candidate_dims = set(candidate_var.dims)
                            if (any(dim in candidate_dims for dim in ['lat', 'latitude']) and 
                                any(dim in candidate_dims for dim in ['lon', 'longitude'])):
                                data_var = candidate_var
                                var_code = name
                                break
                    
                    # Last resort: find any variable with proper spatial dimensions
                    if data_var is None:
                        excluded_vars = ['time_bnds', 'lat_bnds', 'lon_bnds', 'bnds', 'bounds']
                        for name in ds.data_vars:
                            if not any(excl in name.lower() for excl in excluded_vars):
                                candidate_var = ds[name]
                                candidate_dims = set(candidate_var.dims)
                                if (any(dim in candidate_dims for dim in ['lat', 'latitude']) and 
                                    any(dim in candidate_dims for dim in ['lon', 'longitude'])):
                                    data_var = candidate_var
                                    var_code = name
                                    print(f"  Warning: Using {var_code} for {variable_name}")
                                    break
            
            if data_var is None:
                raise ValueError(f"No suitable climate variable found for {variable_name}")
            
            # Apply unit conversions
            data_var, conversion_info = self._apply_unit_conversion(data_var, variable_name, var_code)
            
            # Store dataset
            self.datasets[dataset_name][variable_name] = data_var
            self.conversion_log[dataset_name][variable_name] = conversion_info
            self.conversion_log[dataset_name][variable_name]['detected_type'] = detected_type
            
            # Print upload summary
            print(f"  ✓ Variable: {data_var.name}")
            print(f"  ✓ Shape: {data_var.shape}")
            print(f"  ✓ Dataset type: {detected_type.upper()}")
            print(f"  ✓ Conversion: {conversion_info['description']}")
            print(f"  ✓ Final units: {conversion_info['final_units']}")
            
            return True
            
        except Exception as e:
            print(f"  ✗ Error uploading {variable_name}: {str(e)}")
            return False
    
    def upload_multiple_files(self, dataset_name, file_mapping):
        """
        Upload multiple NetCDF files for a dataset
        
        Parameters:
        -----------
        dataset_name : str
            Name for this dataset
        file_mapping : dict
            Dictionary mapping variable names to file paths
            e.g., {'temperature': '/path/to/tas.nc', 'precipitation': '/path/to/pr.nc'}
            
        Returns:
        --------
        bool : True if all files uploaded successfully
        """
        print(f"\n{'='*70}")
        print(f"UPLOADING FILES FOR JJA {dataset_name.upper()} ANALYSIS")
        print(f"{'='*70}")
        
        success_count = 0
        total_files = len(file_mapping)
        
        for var_name, file_path in file_mapping.items():
            if self.upload_file(dataset_name, var_name, file_path):
                success_count += 1
        
        print(f"\n{'='*70}")
        print(f"Upload Summary: {success_count}/{total_files} files uploaded successfully")
        print(f"{'='*70}")
        
        return success_count == total_files


    def _standardize_coordinates(self, dataset_name):
        """
        Standardize coordinate names to lat/lon/time
        Handles various naming conventions (latitude/Latitude/LAT, etc.)
        """
        print(f"Standardizing coordinates for {dataset_name}...")
        
        standardized_datasets = {}
        
        for var_name, ds in self.datasets[dataset_name].items():
            ds_std = ds.copy()
            coord_mapping = {}
            
            # Time coordinate
            for coord in ['Time', 'T']:
                if coord in ds_std.coords and coord != 'time':
                    coord_mapping[coord] = 'time'
            
            # Latitude coordinate  
            for coord in ['latitude', 'Latitude', 'LAT', 'y']:
                if coord in ds_std.coords and coord != 'lat':
                    coord_mapping[coord] = 'lat'
            
            # Longitude coordinate
            for coord in ['longitude', 'Longitude', 'LON', 'x']:
                if coord in ds_std.coords and coord != 'lon':
                    coord_mapping[coord] = 'lon'
            
            # Apply renaming
            if coord_mapping:
                ds_std = ds_std.rename(coord_mapping)
                print(f"  {var_name}: Renamed {coord_mapping}")
            
            # Ensure longitude is in [-180, 180] range
            if 'lon' in ds_std.coords:
                lon_vals = ds_std.lon.values
                if np.any(lon_vals > 180):
                    ds_std = ds_std.assign_coords(lon=(ds_std.lon + 180) % 360 - 180)
                    ds_std = ds_std.sortby('lon')
                    print(f"  {var_name}: Converted longitude to [-180, 180]")
            
            standardized_datasets[var_name] = ds_std
        
        self.datasets[dataset_name] = standardized_datasets
        return True
    
    def _create_jja_seasons_fixed(self, data_var, start_year, end_year):
        """
        Create JJA seasonal means from monthly data
        
        ⭐ JJA season definition (DIFFERENT FROM DJF!):
        - JJA 2021 = Jun 2021 + Jul 2021 + Aug 2021
        - JJA 2022 = Jun 2022 + Jul 2022 + Aug 2022
        - etc.
        
        Note: JJA is simpler than DJF - all months from SAME year!
        
        Handles both cftime objects and standard datetime
        """
        try:
            import xarray as xr
            
            time_vals = data_var.time.values
            
            # Handle cftime objects
            if hasattr(time_vals[0], '__class__') and 'cftime' in str(type(time_vals[0])):
                print(f"    Detected cftime objects")
                years = np.array([t.year for t in time_vals])
                months = np.array([t.month for t in time_vals])
            else:
                print(f"    Standard datetime objects")
                time_pd = pd.to_datetime(time_vals)
                years = time_pd.year.values
                months = time_pd.month.values
            
            # Create time index map
            time_index_map = {}
            for i, (year, month) in enumerate(zip(years, months)):
                time_index_map[(year, month)] = i
            
            # Create JJA seasons
            jja_list = []
            jja_years = []
            
            for year in range(start_year, end_year + 1):
                # For JJA season of year Y:
                # Need Jun(Y), Jul(Y), Aug(Y) - all from SAME year!
                jun_key = (year, 6)   # June = month 6
                jul_key = (year, 7)   # July = month 7
                aug_key = (year, 8)   # August = month 8
                
                if all(key in time_index_map for key in [jun_key, jul_key, aug_key]):
                    # Get indices
                    jun_idx = time_index_map[jun_key]
                    jul_idx = time_index_map[jul_key]
                    aug_idx = time_index_map[aug_key]
                    
                    # Extract months using .isel()
                    jun_data = data_var.isel(time=jun_idx)
                    jul_data = data_var.isel(time=jul_idx)
                    aug_data = data_var.isel(time=aug_idx)
                    
                    # Calculate JJA mean
                    jja_mean = xr.concat([jun_data, jul_data, aug_data], dim='temp_time').mean(dim='temp_time')
                    
                    jja_list.append(jja_mean)
                    jja_years.append(year)
                    
                    print(f"    JJA {year}: Jun{year} + Jul{year} + Aug{year}")
                else:
                    missing = []
                    if jun_key not in time_index_map:
                        missing.append(f"Jun{year}")
                    if jul_key not in time_index_map:
                        missing.append(f"Jul{year}")
                    if aug_key not in time_index_map:
                        missing.append(f"Aug{year}")
                    print(f"    JJA {year}: Missing {', '.join(missing)}")
            
            if not jja_list:
                print("    No JJA seasons could be created")
                return None
            
            # Combine all JJA seasons
            jja_combined = xr.concat(jja_list, dim='time')
            
            # Create time coordinate (middle of JJA = Jul 15)
            new_time_coords = pd.date_range(
                start=f'{jja_years[0]}-07-15',  # Jul 15, not Jan 15!
                periods=len(jja_years),
                freq='AS'
            )
            jja_combined = jja_combined.assign_coords(time=new_time_coords)
            
            print(f"    Created {len(jja_years)} JJA seasons: {jja_years[0]}-{jja_years[-1]}")
            
            return jja_combined
            
        except Exception as e:
            print(f"    Error creating JJA seasons: {str(e)}")
            return None
    
    def process_to_jja_regional(self, dataset_name, start_year=2071, end_year=2100):
        """
        Process uploaded data to JJA seasonal regional means
        
        Steps:
        1. Standardize coordinates
        2. Create JJA seasonal means for each variable
        3. Extract regional averages (area-weighted)
        4. Create pandas DataFrames indexed by JJA year
        
        Parameters:
        -----------
        dataset_name : str
            Name of dataset to process
        start_year : int
            First JJA year to create (default: 2021)
        end_year : int
            Last JJA year to create (default: 2050)
        """
        if dataset_name not in self.datasets or not self.datasets[dataset_name]:
            print(f"No data uploaded for {dataset_name}!")
            return False
        
        print(f"\n{'='*75}")
        print(f"PROCESSING {dataset_name.upper()} TO JJA SEASONAL REGIONAL MEANS")
        print(f"{'='*75}")
        print(f"JJA seasons to create: {start_year}-{end_year} ({end_year-start_year+1} seasons)")
        
        # Standardize coordinates
        self._standardize_coordinates(dataset_name)
        
        # Initialize storage for regional data
        regional_dataframes = {region: {} for region in self.regions.keys()}
        
        # Process each variable
        for var_name, ds in self.datasets[dataset_name].items():
            print(f"\n--- Processing {var_name} ---")
            
            try:
                # Create JJA seasonal means
                jja_seasons = self._create_jja_seasons_fixed(ds, start_year, end_year)
                
                if jja_seasons is None:
                    print(f"  Error: Could not create JJA seasons for {var_name}")
                    continue
                
                print(f"  JJA shape: {jja_seasons.shape}")
                
                # Extract regional means
                for region_name, bounds in self.regions.items():
                    try:
                        if 'lat' in jja_seasons.dims and 'lon' in jja_seasons.dims:
                            regional_ds = jja_seasons.sel(
                                lat=slice(bounds['lat_min'], bounds['lat_max'])
                            )
                            
                            if len(regional_ds.lat) > 0:
                                # Area-weighted mean
                                weights = np.cos(np.deg2rad(regional_ds.lat))
                                regional_mean = regional_ds.weighted(weights).mean(['lat', 'lon'])
                                regional_dataframes[region_name][var_name] = regional_mean.values
                                
                                print(f"    {region_name}: mean = {np.mean(regional_mean.values):.3f}")
                    
                    except Exception as e:
                        print(f"    {region_name}: Error - {str(e)}")
            
            except Exception as e:
                print(f"  Error processing {var_name}: {str(e)}")
        
        # Convert to pandas DataFrames
        final_regional_data = {}
        jja_years = list(range(start_year, end_year + 1))
        
        for region_name, region_vars in regional_dataframes.items():
            if region_vars:
                df = pd.DataFrame(region_vars)
                
                # Set index to JJA years
                if len(df) <= len(jja_years):
                    df.index = jja_years[:len(df)]
                else:
                    df.index = jja_years
                    df = df.iloc[:len(jja_years)]
                
                df.index.name = 'jja_year'  # ← JJA year, not DJF year!
                final_regional_data[region_name] = df
                
                print(f"\n{region_name}: {df.shape[0]} JJA seasons, {df.shape[1]} variables")
        
        # Store
        if dataset_name not in self.regional_data:
            self.regional_data[dataset_name] = {}
        self.regional_data[dataset_name] = final_regional_data
        
        if final_regional_data:
            print(f"\n✓ Processed {len(final_regional_data)} regions for {dataset_name}")
            return True
        else:
            print(f"\n✗ No regional data processed for {dataset_name}")
            return False
            

    def _normalize_feedback_strength(self, feedback_strengths):
        """
        Normalize feedback strength values to 0-1 scale
        Uses Min-Max normalization across all variables
        
        Parameters:
        -----------
        feedback_strengths : dict
            Dictionary of {variable: feedback_strength}
            
        Returns:
        --------
        tuple : (normalized_strengths, min_val, max_val)
        """
        if not feedback_strengths:
            return feedback_strengths, 0, 0
        
        values = np.array(list(feedback_strengths.values()))
        if len(values) == 0:
            return feedback_strengths, 0, 0
        
        # Min-Max normalization to [0, 1]
        min_val = np.min(values)
        max_val = np.max(values)
        
        if max_val == min_val:
            normalized_values = np.full_like(values, 0.5)
        else:
            normalized_values = (values - min_val) / (max_val - min_val)
        
        # Create normalized dictionary
        normalized_strengths = {}
        for i, var in enumerate(feedback_strengths.keys()):
            normalized_strengths[var] = normalized_values[i]
        
        return normalized_strengths, min_val, max_val
    
    def _block_bootstrap_sample(self, n, block_length=5):
        """
        Generate block bootstrap indices for time series
        
        This is MOVING BLOCK BOOTSTRAP - appropriate for time series!
        
        Parameters:
        -----------
        n : int
            Length of original time series
        block_length : int
            Length of each block (rule of thumb: n^(1/3))
            For 30 JJA seasons: block_length = 3-5 recommended
        
        Returns:
        --------
        bootstrap_indices : array
            Indices for bootstrap sample that preserve temporal structure
            
        How it works:
        -------------
        Instead of resampling individual observations (which breaks autocorrelation),
        we resample BLOCKS of consecutive observations. This preserves the temporal
        dependence structure within each block while still allowing variability
        between bootstrap samples.
        
        Example with n=10, block_length=3:
        - Original: [0,1,2,3,4,5,6,7,8,9]
        - Sample blocks starting at positions: [1, 5, 2, 7]
        - Bootstrap sample: [1,2,3, 5,6,7, 2,3,4, 7,8,9] (first 10 values)
        """
        n_blocks = int(np.ceil(n / block_length))
        
        # Sample block starting positions with replacement
        # Can only start blocks where they won't go past the end
        max_start = n - block_length + 1
        block_starts = np.random.choice(
            max_start, 
            size=n_blocks, 
            replace=True  # With replacement - this is bootstrap!
        )
        
        # Construct bootstrap sample from blocks
        bootstrap_indices = []
        for start in block_starts:
            # Add all indices in this block
            bootstrap_indices.extend(range(start, min(start + block_length, n)))
        
        # Truncate to original length
        return np.array(bootstrap_indices[:n])
    
    def _shap_analysis(self, rf_model, X_train, X_test, feature_names, target_var):
        """
        Perform SHAP analysis on trained Random Forest model
        
        Returns SHAP values, feature importance, variable dependencies,
        directional effects, and feature interactions
        
        Parameters:
        -----------
        rf_model : RandomForestRegressor
            Fitted Random Forest model
        X_train : DataFrame
            Training features
        X_test : DataFrame
            Test features
        feature_names : list
            List of feature names
        target_var : str
            Name of target variable
            
        Returns:
        --------
        dict : SHAP results including:
            - shap_values: SHAP values for test set
            - shap_importance: Mean absolute SHAP values
            - variable_dependencies: How much each variable contributes
            - directional_effects: Positive/negative influence
            - top_interactions: Feature interaction strengths
        """
        if not SHAP_AVAILABLE:
            return None
        
        print(f"    Running SHAP analysis for {target_var}...")
        
        try:
            explainer = shap.TreeExplainer(rf_model)
            
            # Sample for efficiency
            test_sample = X_test.iloc[:min(50, len(X_test))]
            train_sample = X_train.iloc[:min(100, len(X_train))]
            
            shap_values = explainer.shap_values(test_sample)
            shap_values_train = explainer.shap_values(train_sample)
            
            # Mean absolute SHAP values (feature importance)
            mean_abs_shap = np.abs(shap_values).mean(axis=0)
            shap_importance = dict(zip(feature_names, mean_abs_shap))
            
            # Variable dependencies
            variable_dependencies = {}
            base_target = target_var.split('_')[0] if '_' in target_var else target_var
            other_vars = [v for v in ['temperature', 'precipitable_water', 'precipitation', 
                                     'evaporation', 'soil_moisture', 'runoff'] if v != base_target]
            
            for var in other_vars:
                var_features = [i for i, feat in enumerate(feature_names) if feat.startswith(var)]
                if var_features:
                    var_dependency = np.abs(shap_values[:, var_features]).sum(axis=1).mean()
                    variable_dependencies[var] = var_dependency
            
            # Directional effects (using correlation method)
            directional_effects = {}
            directional_effects_detailed = {}
            
            for var in other_vars:
                var_features = [i for i, feat in enumerate(feature_names) if feat.startswith(var)]
                
                if var_features:
                    current_feature = f'{var}_current'
                    
                    if current_feature in feature_names:
                        current_idx = feature_names.index(current_feature)
                        median_shap = np.median(shap_values[:, current_idx])
                        
                        # Correlation between feature value and SHAP value
                        feature_values = test_sample.iloc[:, current_idx].values
                        feature_shap_values = shap_values[:, current_idx]
                        
                        if len(feature_values) > 3:
                            correlation = np.corrcoef(feature_values, feature_shap_values)[0, 1]
                            var_effect = correlation
                            
                            directional_effects_detailed[var] = {
                                'correlation': correlation,
                                'median_shap': median_shap,
                                'interpretation': 'increases' if correlation > 0 else 'decreases'
                            }
                        else:
                            var_effect = median_shap
                            directional_effects_detailed[var] = {
                                'median_shap': median_shap,
                                'interpretation': 'increases' if median_shap > 0 else 'decreases'
                            }
                    else:
                        all_shap = shap_values[:, var_features]
                        var_effect = np.median(all_shap)
                        directional_effects_detailed[var] = {
                            'median_shap_aggregate': var_effect,
                            'interpretation': 'increases' if var_effect > 0 else 'decreases'
                        }
                    
                    directional_effects[var] = var_effect
            
            # Feature interactions
            try:
                top_features_idx = np.argsort(mean_abs_shap)[-6:]
                interaction_sample = test_sample.iloc[:min(20, len(test_sample))]
                interaction_values = explainer.shap_interaction_values(interaction_sample)
                
                interactions = {}
                for i, idx1 in enumerate(top_features_idx):
                    for j, idx2 in enumerate(top_features_idx):
                        if i < j:
                            strength = np.abs(interaction_values[:, idx1, idx2]).mean()
                            interactions[f"{feature_names[idx1]}_x_{feature_names[idx2]}"] = strength
                
                top_interactions = dict(sorted(interactions.items(), key=lambda x: x[1], reverse=True)[:5])
            except:
                top_interactions = {}
            
            print(f"    SHAP analysis completed")
            
            return {
                'shap_values': shap_values,
                'shap_values_train': shap_values_train,
                'explainer': explainer,
                'shap_importance': shap_importance,
                'variable_dependencies': variable_dependencies,
                'directional_effects': directional_effects,
                'directional_effects_detailed': directional_effects_detailed,
                'top_interactions': top_interactions,
                'mean_abs_shap': mean_abs_shap,
                'test_sample': test_sample,
                'train_sample': train_sample
            }
            
        except Exception as e:
            print(f"    SHAP error: {e}")
            return None
    
    def _compare_shap_granger_results(self, dataset_name, region_name, rf_results, gc_detailed):
        """
        Compare SHAP dependencies with Granger causality results
        
        Categories relationships as:
        - BOTH_STRONG: Both methods find strong relationship
        - GC_ONLY: Granger finds it, SHAP doesn't
        - SHAP_ONLY: SHAP finds it, Granger doesn't
        - BOTH_WEAK: Neither method finds strong relationship
        """
        if not SHAP_AVAILABLE:
            return {}
        
        print(f"\nCOMPARING SHAP VS GRANGER CAUSALITY")
        print("-" * 50)
        
        comparison_results = {}
        
        for target_var in rf_results.keys():
            if 'shap_results' not in rf_results[target_var] or not rf_results[target_var]['shap_results']:
                continue
            
            shap_data = rf_results[target_var]['shap_results']
            var_dependencies = shap_data.get('variable_dependencies', {})
            directional_effects = shap_data.get('directional_effects', {})
            directional_details = shap_data.get('directional_effects_detailed', {})
            
            if not var_dependencies:
                continue
            
            target_comparisons = {}
            
            for source_var in var_dependencies.keys():
                shap_dependency = var_dependencies[source_var]
                shap_direction = directional_effects.get(source_var, 0)
                
                interpretation = directional_details.get(source_var, {}).get('interpretation', 
                                                        'increases' if shap_direction > 0 else 'decreases')
                
                # Granger result
                gc_key = f"{source_var}_causes_{target_var}"
                granger_result = gc_detailed.get(gc_key, {})
                
                gc_p_value = granger_result.get('primary_p_value', 1.0)
                gc_significant = granger_result.get('primary_is_significant', False)
                gc_lag = granger_result.get('primary_lag', 2)
                
                # Categorize agreement
                shap_strong = shap_dependency > np.mean(list(var_dependencies.values()))
                
                if gc_significant and shap_strong:
                    agreement = "BOTH_STRONG"
                elif gc_significant and not shap_strong:
                    agreement = "GC_ONLY"
                elif not gc_significant and shap_strong:
                    agreement = "SHAP_ONLY"
                else:
                    agreement = "BOTH_WEAK"
                
                target_comparisons[source_var] = {
                    'shap_dependency': shap_dependency,
                    'shap_direction': shap_direction,
                    'shap_interpretation': interpretation,
                    'gc_p_value': gc_p_value,
                    'gc_significant': gc_significant,
                    'gc_lag': gc_lag,
                    'agreement': agreement
                }
            
            comparison_results[target_var] = target_comparisons
        
        return comparison_results


    def _enhanced_random_forest_analysis_with_shap(
        self, data, variables, *,
        dataset_name: str,
        region_name: str,
        make_shap_plots: bool = False):
        """
        Enhanced Random Forest analysis with SHAP + Block Bootstrap for JJA seasons
        
        Creates exactly 19 features per target variable:
        - 4 autoregressive lags (target variable)
        - 15 feedback features (5 other variables × 3 each)
        
        Process:
        1. Create feature matrix with lags
        2. Train-test split (temporal)
        3. Fit baseline model (autoregressive only)
        4. Fit full model (autoregressive + feedback)
        5. SHAP analysis (if enabled)
        6. Block bootstrap uncertainty (if enabled)
        
        Returns:
        --------
        dict : Results for each target variable including:
            - r2_baseline, r2_full
            - feedback_strength
            - feature_importance
            - predictions
            - shap_results (if SHAP enabled)
            - bootstrap_stats (if bootstrap enabled)
        """
        from sklearn.model_selection import train_test_split
        
        rf_results = {}
        
        print(f"Creating 19 features per variable: 4 AR + 15 feedback")
        
        for target_var in variables:
            print(f"\n  Analyzing feedback TO {target_var}...")
            
            # ================================================================
            # INITIALIZE ALL VARIABLES (prevents scope errors)
            # ================================================================
            predictions = None
            shap_results = None
            bootstrap_stats = None
            r2_baseline = 0.0
            r2_full = 0.0
            feedback_strength = 0.0
            feature_importance = {}
            
            # ================================================================
            # CREATE FEATURE MATRIX (19 features)
            # ================================================================
            feature_matrix = pd.DataFrame(index=data.index)
            
            # AUTOREGRESSIVE: 4 lags
            autoregressive_features = []
            for lag in range(1, 5):
                lag_col = f'{target_var}_lag_{lag}'
                feature_matrix[lag_col] = data[target_var].shift(lag)
                autoregressive_features.append(lag_col)
            
            # FEEDBACK: 15 features (5 other vars × 3 each)
            feedback_features = []
            other_vars = [v for v in variables if v != target_var]
            
            for var in other_vars:
                # Current value
                feature_matrix[f'{var}_current'] = data[var]
                feedback_features.append(f'{var}_current')
                
                # 2 lags
                for lag in [1, 2]:
                    lag_col = f'{var}_lag_{lag}'
                    feature_matrix[lag_col] = data[var].shift(lag)
                    feedback_features.append(lag_col)
            
            # Verify counts
            n_autoregressive = len(autoregressive_features)  # = 4
            n_feedback = len(feedback_features)              # = 15
            n_total = n_autoregressive + n_feedback          # = 19
            
            print(f"    Features: {n_autoregressive} AR + {n_feedback} feedback = {n_total} total")
            
            # Remove NaN rows
            feature_matrix = feature_matrix.dropna()
            y_target = data[target_var].loc[feature_matrix.index]
            
            if len(feature_matrix) < 8:
                print(f"    Insufficient data: {len(feature_matrix)} samples")
                rf_results[target_var] = {
                    'r2_baseline': 0.0, 'r2_full': 0.0, 'feedback_strength': 0.0,
                    'feature_importance': {}, 'predictions': None, 'shap_results': None,
                    'bootstrap_stats': None, 'n_samples': len(feature_matrix),
                    'n_features_total': n_total, 'error': 'insufficient_data'
                }
                continue
            
            print(f"    Available samples: {len(feature_matrix)}")
            
            # Define feature sets
            X_baseline = feature_matrix[autoregressive_features]
            all_features = autoregressive_features + feedback_features
            X_full = feature_matrix[all_features]
            
            # Temporal split
            test_size = max(0.25, min(0.4, 8 / len(feature_matrix)))
            split_point = int((1 - test_size) * len(feature_matrix))
            split_point = max(5, split_point)
            
            # ================================================================
            # MAIN TRY BLOCK - ALL CODE BELOW IS INSIDE THIS TRY
            # ================================================================
            try:
                # Split data
                X_base_train = X_baseline.iloc[:split_point]
                X_base_test = X_baseline.iloc[split_point:]
                X_full_train = X_full.iloc[:split_point]
                X_full_test = X_full.iloc[split_point:]
                y_train = y_target.iloc[:split_point]
                y_test = y_target.iloc[split_point:]
                
                print(f"    Train: {len(X_base_train)}, Test: {len(X_base_test)}")
                
                # ============================================================
                # STEP 1: BASELINE MODEL (autoregressive only)
                # ============================================================
                rf_baseline = RandomForestRegressor(
                    n_estimators=200, max_depth=3, min_samples_leaf=2,
                    min_samples_split=4, max_features='sqrt',
                    random_state=self.random_state
                )
                
                if len(X_base_train) >= 4:
                    rf_baseline.fit(X_base_train, y_train)
                    
                    if len(y_test) > 0:
                        y_pred_baseline = rf_baseline.predict(X_base_test)
                        r2_baseline = max(0.0, r2_score(y_test, y_pred_baseline))
                
                # ============================================================
                # STEP 2: FULL MODEL (autoregressive + feedback)
                # ============================================================
                rf_full = RandomForestRegressor(
                    n_estimators=200, max_depth=3, min_samples_leaf=3,
                    min_samples_split=6, max_features=min(8, len(all_features)//2),
                    random_state=self.random_state
                )
                
                if len(X_full_train) >= 4:
                    rf_full.fit(X_full_train, y_train)
                    
                    if len(y_test) > 0:
                        y_pred_full = rf_full.predict(X_full_test)
                        r2_full = max(0.0, r2_score(y_test, y_pred_full))
                        
                        rmse = np.sqrt(mean_squared_error(y_test, y_pred_full))
                        mae = mean_absolute_error(y_test, y_pred_full)
                        
                        predictions = {
                            'y_test': y_test.values,
                            'y_pred': y_pred_full,
                            'r2_score': r2_full,
                            'rmse': rmse,
                            'mae': mae,
                            'n_test_samples': len(y_test)
                        }
                        
                        # ====================================================
                        # STEP 3: SHAP ANALYSIS (if enabled)
                        # ====================================================
                        if SHAP_AVAILABLE and len(X_full_test) > 0:
                            shap_results = self._shap_analysis(
                                rf_full, X_full_train, X_full_test, all_features, target_var
                            )
                            
                            if shap_results:
                                shap_results['rf_feature_importance'] = dict(zip(
                                    all_features, rf_full.feature_importances_
                                ))
                    else:
                        predictions = None
                        shap_results = None
                else:
                    r2_full = r2_baseline
                    predictions = None
                    shap_results = None
                
                # ============================================================
                # STEP 4: EXTRACT FEATURE IMPORTANCE
                # ============================================================
                try:
                    feature_importance = dict(zip(all_features, rf_full.feature_importances_))
                    print(f"    Extracted {len(feature_importance)} feature importances")
                except:
                    feature_importance = {}
                
                # ============================================================
                # STEP 5: CALCULATE FEEDBACK STRENGTH
                # ============================================================
                feedback_strength = max(0, r2_full - r2_baseline)
                
                # ============================================================
                # STEP 6: BLOCK BOOTSTRAP (OPTIONAL)
                # ============================================================
                if hasattr(self, 'n_bootstrap') and self.n_bootstrap > 0:
                    print(f"    Running block bootstrap ({self.n_bootstrap} samples)...")
                    
                    # Optimal block length
                    block_length = max(3, int(len(X_full_train) ** (1/3)))
                    print(f"    Block length: {block_length} seasons")
                    
                    bootstrap_results = {
                        'feedback_strength': [],
                        'r2_full': [],
                        'r2_baseline': []
                    }
                    
                    successful = 0
                    for b in range(self.n_bootstrap):
                        if b % 200 == 0:
                            print(f"      Iteration {b}/{self.n_bootstrap}")
                        
                        # Block bootstrap resample
                        boot_indices = self._block_bootstrap_sample(len(X_full_train), block_length)
                        
                        X_base_boot = X_base_train.iloc[boot_indices]
                        X_full_boot = X_full_train.iloc[boot_indices]
                        y_boot = y_train.iloc[boot_indices]
                        
                        try:
                            # Baseline
                            rf_base_boot = RandomForestRegressor(
                                n_estimators=100, max_depth=3, min_samples_leaf=2,
                                random_state=self.random_state + b
                            )
                            rf_base_boot.fit(X_base_boot, y_boot)
                            y_pred_base = rf_base_boot.predict(X_base_test)
                            r2_base_boot = max(0, r2_score(y_test, y_pred_base))
                            
                            # Full
                            rf_full_boot = RandomForestRegressor(
                                n_estimators=100, max_depth=3, min_samples_leaf=3,
                                random_state=self.random_state + b
                            )
                            rf_full_boot.fit(X_full_boot, y_boot)
                            y_pred_full_boot = rf_full_boot.predict(X_full_test)
                            r2_full_boot = max(0, r2_score(y_test, y_pred_full_boot))
                            
                            # Store
                            bootstrap_results['r2_baseline'].append(r2_base_boot)
                            bootstrap_results['r2_full'].append(r2_full_boot)
                            bootstrap_results['feedback_strength'].append(max(0, r2_full_boot - r2_base_boot))
                            successful += 1
                        except:
                            continue
                    
                    print(f"    Successful: {successful}/{self.n_bootstrap}")
                    
                    # Calculate statistics
                    if len(bootstrap_results['feedback_strength']) > 0:
                        bootstrap_stats = {
                            'feedback_strength': {
                                'mean': np.mean(bootstrap_results['feedback_strength']),
                                'std': np.std(bootstrap_results['feedback_strength']),
                                'ci_lower': np.percentile(bootstrap_results['feedback_strength'], 2.5),
                                'ci_upper': np.percentile(bootstrap_results['feedback_strength'], 97.5),
                                'distribution': bootstrap_results['feedback_strength']
                            },
                            'r2_baseline': {
                                'mean': np.mean(bootstrap_results['r2_baseline']),
                                'std': np.std(bootstrap_results['r2_baseline']),
                                'ci_lower': np.percentile(bootstrap_results['r2_baseline'], 2.5),
                                'ci_upper': np.percentile(bootstrap_results['r2_baseline'], 97.5)
                            },
                            'r2_full': {
                                'mean': np.mean(bootstrap_results['r2_full']),
                                'std': np.std(bootstrap_results['r2_full']),
                                'ci_lower': np.percentile(bootstrap_results['r2_full'], 2.5),
                                'ci_upper': np.percentile(bootstrap_results['r2_full'], 97.5)
                            },
                            'n_successful': successful,
                            'block_length': block_length
                        }
                        
                        bs_fb = bootstrap_stats['feedback_strength']
                        print(f"    Bootstrap feedback: {bs_fb['mean']:.4f} ± {bs_fb['std']:.4f}")
                        print(f"    95% CI: [{bs_fb['ci_lower']:.4f}, {bs_fb['ci_upper']:.4f}]")
                        
                        if bs_fb['ci_lower'] > 0:
                            print(f"    *** STATISTICALLY SIGNIFICANT ***")
                    else:
                        bootstrap_stats = None
            
            # ================================================================
            # EXCEPTION HANDLER (catches errors from the try block above)
            # ================================================================
            except Exception as e:
                print(f"    ERROR in RF analysis: {e}")
                
                # Set safe defaults
                r2_baseline = 0.0
                r2_full = 0.0
                feedback_strength = 0.0
                feature_importance = {}
                predictions = None
                shap_results = None
                bootstrap_stats = None
            
            # ================================================================
            # STORE RESULTS (ALWAYS happens, outside try/except)
            # ================================================================
            rf_results[target_var] = {
                'r2_baseline': r2_baseline,
                'r2_full': r2_full,
                'feedback_strength': feedback_strength,
                'feature_importance': feature_importance,
                'predictions': predictions,
                'shap_results': shap_results,
                'bootstrap_stats': bootstrap_stats,
                'n_samples': len(feature_matrix),
                'n_features_baseline': n_autoregressive,
                'n_features_full': n_total,
                'feature_design': 'exactly_19_jja',  # ← ONLY DIFFERENCE FROM DJF!
                'validation_method': 'temporal_holdout_with_block_bootstrap' if bootstrap_stats else 'temporal_holdout'
            }
            
            # Print summary
            print(f"    R² baseline: {r2_baseline:.4f}")
            print(f"    R² full: {r2_full:.4f}")
            print(f"    Feedback strength: {feedback_strength:.4f}")
            print(f"    SHAP: {shap_results is not None}")
            print(f"    Bootstrap: {bootstrap_stats is not None}")
        
        return rf_results


    def _granger_causality_analysis(self, data, variables):
        """
        Granger causality analysis for directional feedback in JJA seasons
        
        Tests: Does past X help predict future Y?
        
        Lag structure (matches Random Forest):
        - Primary lag 2: Matches RF feedback features (current + lags 1-2)
        - Max lag 4: Matches RF autoregressive features (lags 1-4)
        
        Parameters:
        -----------
        data : DataFrame
            Preprocessed JJA seasonal data
        variables : list
            List of variable names to test
            
        Returns:
        --------
        tuple : (causality_matrix, detailed_results)
            - causality_matrix: DataFrame of p-values (primary lag 2)
            - detailed_results: dict with p-values for all tested lags
        """
        if len(variables) < 2:
            return None, None
        
        feedback_lag = 2  # Primary lag (matches RF feedback)
        max_ar_lag = 4    # Maximum lag to test (matches RF autoregressive)
        
        print(f"\nGRANGER CAUSALITY TESTING")
        print(f"  Primary lag: {feedback_lag} (matches RF feedback)")
        print(f"  Max lag: {max_ar_lag} (matches RF autoregressive)")
        
        # Initialize results storage
        causality_matrix = pd.DataFrame(
            np.nan,
            index=variables,
            columns=variables
        )
        
        detailed_results = {}
        significant_links = []
        
        # Test all variable pairs
        for cause_var in variables:
            for effect_var in variables:
                if cause_var == effect_var:
                    continue
                
                test_data = data[[effect_var, cause_var]].dropna()
                
                if len(test_data) < max_ar_lag + 5:
                    continue
                
                try:
                    # Run Granger causality test
                    gc_result = grangercausalitytests(
                        test_data, 
                        max_ar_lag, 
                        verbose=False
                    )
                    
                    # Extract p-values for all lags
                    p_values_all_lags = {}
                    for lag in range(1, max_ar_lag + 1):
                        if lag in gc_result:
                            p_val = gc_result[lag][0]['ssr_ftest'][1]
                            p_values_all_lags[lag] = p_val
                    
                    if not p_values_all_lags:
                        continue
                    
                    # PRIMARY RESULT: Use lag 2
                    primary_p_value = p_values_all_lags.get(feedback_lag, 1.0)
                    primary_significant = primary_p_value <= self.significance_level
                    
                    # SECONDARY: Find best lag
                    min_p_value = min(p_values_all_lags.values())
                    best_lag = min(p_values_all_lags.keys(), key=lambda k: p_values_all_lags[k])
                    best_significant = min_p_value <= self.significance_level
                    
                    # Store in matrix (primary lag 2)
                    causality_matrix.loc[cause_var, effect_var] = primary_p_value
                    
                    # Store detailed results
                    detailed_results[f"{cause_var}_causes_{effect_var}"] = {
                        'primary_p_value': primary_p_value,
                        'primary_lag': feedback_lag,
                        'primary_is_significant': primary_significant,
                        'min_p_value': min_p_value,
                        'best_lag': best_lag,
                        'best_is_significant': best_significant,
                        'all_lag_p_values': p_values_all_lags
                    }
                    
                    # Report significant relationships
                    if primary_significant:
                        stars = "***" if primary_p_value <= 0.001 else "**" if primary_p_value <= 0.01 else "*"
                        print(f"  {cause_var} → {effect_var}: p={primary_p_value:.4f} {stars}")
                        significant_links.append(f"{cause_var} → {effect_var}")
                        
                        if best_lag != feedback_lag and best_significant:
                            print(f"    (Best at lag {best_lag}: p={min_p_value:.4f})")
                
                except Exception as e:
                    print(f"  Error: {cause_var} → {effect_var}: {str(e)}")
        
        # Summary
        if significant_links:
            print(f"\nFound {len(significant_links)} significant causal relationships")
        else:
            print(f"\nNo significant causal relationships found (p > {self.significance_level})")
        
        return causality_matrix, detailed_results


    def run_mutual_feedback_analysis(self, dataset_name, region_name):
        """
        Run complete mutual feedback analysis for one dataset and region (JJA seasons)
        
        This is the MAIN orchestration method that:
        1. Checks data availability
        2. Preprocesses data (stationarity testing)
        3. Runs Random Forest analysis with SHAP
        4. Normalizes feedback strengths
        5. Runs Granger causality analysis
        6. Compares SHAP vs Granger results
        """
        if (dataset_name not in self.regional_data or 
            region_name not in self.regional_data[dataset_name]):
            print(f"No JJA data available for {dataset_name} - {region_name}")
            return None
        
        data = self.regional_data[dataset_name][region_name]
        
        print(f"\n{'='*70}")
        print(f"RUNNING JJA MUTUAL FEEDBACK ANALYSIS")
        print(f"Dataset: {dataset_name.upper()}, Region: {region_name}")
        print(f"{'='*70}")
        
        available_vars = [col for col in data.columns 
                         if col in list(self.climate_variables.keys())]
        
        if len(available_vars) < 2:
            print(f"Need at least 2 variables, found {len(available_vars)}")
            return None
        
        print(f"Variables: {available_vars}")
        print(f"JJA seasons: {data.index[0]}-{data.index[-1]} ({len(data)} total)")
        
        # STEP 1: PREPROCESS
        processed_data = data[available_vars].copy()
        transformations = {}
        
        print(f"\n1. CHECKING STATIONARITY...")
        for var in available_vars:
            series = processed_data[var].dropna()
            if len(series) > 8:
                adf_result = adfuller(series)
                is_stationary = adf_result[1] <= self.significance_level
                
                print(f"  {var}: p={adf_result[1]:.4f}, stationary={is_stationary}")
                
                if not is_stationary and len(series) > 10:
                    diff_series = series.diff().dropna()
                    if len(diff_series) > 5:
                        processed_data[var] = series.diff()
                        transformations[var] = 'first_difference'
                        print(f"    Applied first differencing")
                    else:
                        transformations[var] = 'none'
                else:
                    transformations[var] = 'none'
            else:
                transformations[var] = 'none'
        
        processed_data = processed_data.dropna()
        
        if len(processed_data) < 10:
            print(f"Insufficient data: {len(processed_data)} seasons")
            return None
        
        print(f"Preprocessed: {len(processed_data)} seasons available")
        
        # STEP 2: RANDOM FOREST + SHAP
        print(f"\n2. RANDOM FOREST FEEDBACK ANALYSIS...")
        rf_results = self._enhanced_random_forest_analysis_with_shap(
            processed_data, available_vars,
            dataset_name=dataset_name,
            region_name=region_name,
            make_shap_plots=False
        )
        
        # STEP 3: NORMALIZE
        print(f"\n3. NORMALIZING FEEDBACK STRENGTHS...")
        raw_strengths = {var: rf_results[var]['feedback_strength'] 
                        for var in rf_results.keys() 
                        if 'feedback_strength' in rf_results[var]}
        
        if raw_strengths:
            normalized_strengths, min_val, max_val = self._normalize_feedback_strength(raw_strengths)
            
            print(f"Raw range: {min_val:.4f} to {max_val:.4f}")
            for var, norm_strength in normalized_strengths.items():
                raw_strength = raw_strengths[var]
                rf_results[var]['feedback_strength_normalized'] = norm_strength
                rf_results[var]['feedback_strength_raw'] = raw_strength
                print(f"  {var}: {raw_strength:.4f} → {norm_strength:.4f}")
        
        # STEP 4: GRANGER CAUSALITY
        print(f"\n4. GRANGER CAUSALITY ANALYSIS...")
        gc_matrix, gc_detailed = self._granger_causality_analysis(processed_data, available_vars)
        
        # STEP 5: SHAP-GRANGER COMPARISON
        if SHAP_AVAILABLE:
            print(f"\n5. SHAP-GRANGER COMPARISON...")
            shap_granger_comparison = self._compare_shap_granger_results(
                dataset_name, region_name, rf_results, gc_detailed
            )
        else:
            shap_granger_comparison = {}
        
        # STORE RESULTS
        feedback_results = {
            'original_data': data,
            'processed_data': processed_data,
            'transformations': transformations,
            'available_variables': available_vars,
            'rf_results': rf_results,
            'gc_matrix': gc_matrix,
            'gc_detailed': gc_detailed,
            'shap_granger_comparison': shap_granger_comparison,
            'dataset_name': dataset_name,
            'region_name': region_name,
            'season': 'JJA',  # ← JJA, not DJF!
            'total_jja_seasons': len(data),  # ← total_jja_seasons, not total_djf_seasons!
            'normalization_info': {
                'raw_min': min_val if raw_strengths else 0,
                'raw_max': max_val if raw_strengths else 0,
                'normalized_strengths': normalized_strengths if raw_strengths else {}
            },
            'shap_enabled': SHAP_AVAILABLE,
            'bootstrap_enabled': self.n_bootstrap > 0
        }
        
        if dataset_name not in self.feedback_results:
            self.feedback_results[dataset_name] = {}
        self.feedback_results[dataset_name][region_name] = feedback_results
        
        print(f"\n✓ Analysis complete for {region_name}")
        
        return feedback_results
    
    def run_feedback_analysis_all_regions(self, dataset_name):
        """Run JJA mutual feedback analysis for ALL regions"""
        print(f"\n{'='*80}")
        print(f"RUNNING JJA ANALYSIS FOR ALL REGIONS")
        print(f"Dataset: {dataset_name.upper()}")
        print(f"{'='*80}")
        
        if dataset_name not in self.regional_data:
            print(f"No data processed for {dataset_name}")
            return None
        
        all_feedback_results = {}
        available_regions = list(self.regional_data[dataset_name].keys())
        
        print(f"Regions to analyze: {len(available_regions)}")
        
        for i, region_name in enumerate(available_regions, 1):
            print(f"\n{'='*80}")
            print(f"REGION {i}/{len(available_regions)}: {region_name.upper()}")
            print(f"{'='*80}")
            
            region_data = self.regional_data[dataset_name][region_name]
            
            if len(region_data) < 15:
                print(f"Insufficient data: {len(region_data)} seasons (need ≥15)")
                continue
            
            if len(region_data.columns) < 2:
                print(f"Need at least 2 variables")
                continue
            
            try:
                feedback_results = self.run_mutual_feedback_analysis(dataset_name, region_name)
                
                if feedback_results:
                    all_feedback_results[region_name] = feedback_results
                    print(f"✓ {region_name} completed")
                else:
                    print(f"✗ {region_name} failed")
            
            except Exception as e:
                print(f"Error: {str(e)}")
        
        print(f"\n{'='*80}")
        print(f"SUMMARY: Analyzed {len(all_feedback_results)}/{len(available_regions)} regions")
        print(f"Successful: {list(all_feedback_results.keys())}")
        print(f"{'='*80}")
        
        return all_feedback_results
    
    def generate_feedback_report(self, dataset_name, region_name):
        """Generate detailed text report of JJA feedback analysis results"""
        if (dataset_name not in self.feedback_results or 
            region_name not in self.feedback_results[dataset_name]):
            print(f"No results for {dataset_name} - {region_name}")
            return None
        
        results = self.feedback_results[dataset_name][region_name]
        rf_results = results['rf_results']
        gc_detailed = results['gc_detailed']
        
        print(f"\n{'='*70}")
        print(f"JJA MUTUAL FEEDBACK ANALYSIS REPORT")
        print(f"{'='*70}")
        print(f"Dataset: {dataset_name.upper()}")
        print(f"Region: {region_name}")
        print(f"Season: JJA (June-July-August)")
        print(f"SHAP: {'ENABLED' if results.get('shap_enabled') else 'DISABLED'}")
        print(f"Bootstrap: {'ENABLED' if results.get('bootstrap_enabled') else 'DISABLED'}")
        print(f"{'='*70}")
        
        print(f"\nData Summary:")
        print(f"  Variables: {results['available_variables']}")
        print(f"  JJA seasons: {results['total_jja_seasons']}")
        print(f"  Transformations: {results['transformations']}")
        
        print(f"\n{'='*70}")
        print(f"RANDOM FOREST FEEDBACK ANALYSIS")
        print(f"{'='*70}")
        
        if rf_results:
            sorted_vars = sorted(rf_results.items(), 
                               key=lambda x: x[1].get('feedback_strength', 0), 
                               reverse=True)
            
            for var, result in sorted_vars:
                print(f"\n{var.upper()}:")
                print(f"  Baseline R² (AR only): {result['r2_baseline']:.4f}")
                print(f"  Full R² (AR + feedback): {result['r2_full']:.4f}")
                print(f"  Feedback strength: {result['feedback_strength']:.4f}")
                
                if 'bootstrap_stats' in result and result['bootstrap_stats']:
                    bs = result['bootstrap_stats']['feedback_strength']
                    print(f"  Bootstrap mean ± std: {bs['mean']:.4f} ± {bs['std']:.4f}")
                    print(f"  95% CI: [{bs['ci_lower']:.4f}, {bs['ci_upper']:.4f}]")
                    
                    if bs['ci_lower'] > 0:
                        print(f"  *** STATISTICALLY SIGNIFICANT ***")
                
                if result.get('shap_results'):
                    shap_data = result['shap_results']
                    var_deps = shap_data.get('variable_dependencies', {})
                    
                    if var_deps:
                        print(f"  SHAP dependencies (top 3):")
                        sorted_deps = sorted(var_deps.items(), key=lambda x: x[1], reverse=True)
                        for dep_var, dep_val in sorted_deps[:3]:
                            direction = shap_data.get('directional_effects', {}).get(dep_var, 0)
                            interp = 'increases' if direction > 0 else 'decreases'
                            print(f"    {dep_var}: {dep_val:.4f} ({interp} target)")
        
        print(f"\n{'='*70}")
        print(f"GRANGER CAUSALITY RESULTS")
        print(f"{'='*70}")
        
        if gc_detailed:
            sig_links = [(link, det) for link, det in gc_detailed.items() 
                        if det.get('primary_is_significant', False)]
            
            if sig_links:
                print(f"Significant relationships: {len(sig_links)}")
                for link, details in sig_links:
                    cause, effect = link.split('_causes_')
                    p = details['primary_p_value']
                    stars = "***" if p <= 0.001 else "**" if p <= 0.01 else "*"
                    print(f"  {cause} → {effect}: p={p:.4f} {stars}")
            else:
                print(f"No significant relationships (p > {self.significance_level})")
        
        print(f"\n{'='*70}")
        print(f"INTERPRETATION GUIDE")
        print(f"{'='*70}")
        
        print(f"\nFeedback Strength:")
        print(f"  0.00-0.10: Weak")
        print(f"  0.10-0.25: Moderate")
        print(f"  0.25-0.50: Strong")
        print(f"  >0.50: Very strong")
        
        print(f"\nR² Performance:")
        print(f"  <0.3: Poor")
        print(f"  0.3-0.6: Moderate")
        print(f"  0.6-0.8: Good")
        print(f"  >0.8: Excellent")
        
        if results.get('bootstrap_enabled'):
            print(f"\nBootstrap Uncertainty:")
            print(f"  • Confidence intervals quantify estimation uncertainty")
            print(f"  • CI excluding zero indicates statistical significance")
            print(f"  • Narrow CI = precise estimate")
            print(f"  • Wide CI = high uncertainty")
        
        if results.get('shap_enabled'):
            print(f"\nSHAP vs Granger:")
            print(f"  • Granger: Linear causality (does past X help predict Y?)")
            print(f"  • SHAP: Any relationship (non-linear included)")
            print(f"  • Both strong: Robust relationship")
            print(f"  • SHAP only: Non-linear dependency")
            print(f"  • Granger only: Linear causality RF misses")
        
        return results
    
    def save_feedback_results(self, output_directory="enhanced_jja_mutual_feedback_results_with_shap"):
        """Save enhanced JJA mutual feedback results with SHAP and Bootstrap - COMPREHENSIVE VERSION"""
        output_path = Path(output_directory)
        output_path.mkdir(parents=True, exist_ok=True)

        print(f"\nSaving enhanced JJA mutual feedback results to {output_path}")

        if not self.feedback_results:
            print("ERROR: No feedback results to save!")
            return False

        total_files_saved = 0

        for dataset_name, dataset_feedback_results in self.feedback_results.items():
            dataset_path = output_path / dataset_name
            dataset_path.mkdir(exist_ok=True)

            for region_name, feedback_results in dataset_feedback_results.items():
                region_path = dataset_path / region_name
                region_path.mkdir(exist_ok=True)
                
                print(f"\nSaving files for {region_name}...")

                # 1. Enhanced RF Results with Bootstrap
                try:
                    if 'rf_results' in feedback_results and feedback_results['rf_results']:
                        rf_data = []
                        for var, results in feedback_results['rf_results'].items():
                            row_data = {
                                'variable': var,
                                'r2_baseline': results.get('r2_baseline', 0.0),
                                'r2_full': results.get('r2_full', 0.0),
                                'feedback_strength_raw': results.get('feedback_strength_raw', results.get('feedback_strength', 0.0)),
                                'feedback_strength_normalized': results.get('feedback_strength_normalized', 0.0),
                                'n_samples': results.get('n_samples', 0),
                                'n_features_baseline': results.get('n_features_baseline', 0),
                                'n_features_full': results.get('n_features_full', 0),
                                'shap_available': results.get('shap_results') is not None
                            }

                            if 'predictions' in results and results['predictions']:
                                pred = results['predictions']
                                row_data.update({
                                    'test_r2': pred.get('r2_score', 0.0),
                                    'test_rmse': pred.get('rmse', 0.0),
                                    'test_mae': pred.get('mae', 0.0),
                                    'n_test_samples': pred.get('n_test_samples', 0)
                                })
                            
                            if 'bootstrap_stats' in results and results['bootstrap_stats']:
                                bs = results['bootstrap_stats']['feedback_strength']
                                row_data.update({
                                    'bootstrap_mean': bs['mean'],
                                    'bootstrap_std': bs['std'],
                                    'bootstrap_ci_lower': bs['ci_lower'],
                                    'bootstrap_ci_upper': bs['ci_upper'],
                                    'statistically_significant': bs['ci_lower'] > 0
                                })
                                
                                if 'r2_full' in results['bootstrap_stats']:
                                    bs_r2 = results['bootstrap_stats']['r2_full']
                                    row_data.update({
                                        'bootstrap_r2_full_mean': bs_r2['mean'],
                                        'bootstrap_r2_full_ci_lower': bs_r2['ci_lower'],
                                        'bootstrap_r2_full_ci_upper': bs_r2['ci_upper']
                                    })

                            rf_data.append(row_data)

                        if rf_data:
                            rf_df = pd.DataFrame(rf_data)
                            rf_df.to_csv(region_path / "enhanced_jja_random_forest_feedback_results_with_shap.csv", index=False)
                            print(f"  Saved: enhanced_jja_random_forest_feedback_results_with_shap.csv")
                            total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving RF results: {e}")

                # 2. SHAP Variable Dependencies (per variable)
                try:
                    if SHAP_AVAILABLE and 'rf_results' in feedback_results:
                        for var, results in feedback_results['rf_results'].items():
                            if 'shap_results' in results and results['shap_results']:
                                shap_data = results['shap_results']
                                
                                var_deps = shap_data.get('variable_dependencies', {})
                                if var_deps:
                                    shap_var_data = []
                                    for source_var, dependency in var_deps.items():
                                        shap_var_data.append({
                                            'target_variable': var,
                                            'source_variable': source_var,
                                            'shap_dependency': dependency,
                                            'directional_effect': shap_data.get('directional_effects', {}).get(source_var, 0)
                                        })
                                    
                                    if shap_var_data:
                                        shap_df = pd.DataFrame(shap_var_data)
                                        shap_df = shap_df.sort_values('shap_dependency', ascending=False)
                                        shap_df.to_csv(region_path / f"jja_shap_variable_dependencies_{var}.csv", index=False)
                                        print(f"  Saved: jja_shap_variable_dependencies_{var}.csv")
                                        total_files_saved += 1
                                
                                shap_importance = shap_data.get('shap_importance', {})
                                if shap_importance:
                                    shap_imp_data = []
                                    for feature, importance in sorted(shap_importance.items(), key=lambda x: x[1], reverse=True):
                                        shap_imp_data.append({
                                            'target_variable': var,
                                            'feature': feature,
                                            'shap_importance': importance
                                        })
                                    
                                    if shap_imp_data:
                                        shap_imp_df = pd.DataFrame(shap_imp_data)
                                        shap_imp_df.to_csv(region_path / f"jja_shap_feature_importance_{var}.csv", index=False)
                                        print(f"  Saved: jja_shap_feature_importance_{var}.csv")
                                        total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving SHAP results: {e}")

                # 3. SHAP-Granger Comparison
                try:
                    if 'shap_granger_comparison' in feedback_results and feedback_results['shap_granger_comparison']:
                        comparison_data = []
                        for target_var, comparisons in feedback_results['shap_granger_comparison'].items():
                            for source_var, comp_data in comparisons.items():
                                comparison_data.append({
                                    'target_variable': target_var,
                                    'source_variable': source_var,
                                    'shap_dependency': comp_data['shap_dependency'],
                                    'shap_direction': comp_data['shap_direction'],
                                    'shap_interpretation': comp_data['shap_interpretation'],
                                    'granger_p_value': comp_data['gc_p_value'],
                                    'granger_significant': comp_data['gc_significant'],
                                    'granger_lag': comp_data['gc_lag'],
                                    'agreement_category': comp_data['agreement']
                                })
                        
                        if comparison_data:
                            comparison_df = pd.DataFrame(comparison_data)
                            comparison_df = comparison_df.sort_values(['agreement_category', 'shap_dependency'], ascending=[True, False])
                            comparison_df.to_csv(region_path / "jja_shap_granger_comparison.csv", index=False)
                            print(f"  Saved: jja_shap_granger_comparison.csv")
                            total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving SHAP-Granger comparison: {e}")

                # 4. RF Feature Importance (per variable)
                try:
                    if 'rf_results' in feedback_results:
                        for var, results in feedback_results['rf_results'].items():
                            if 'feature_importance' in results and results['feature_importance']:
                                importance_data = []
                                
                                sorted_importance = sorted(results['feature_importance'].items(), 
                                                        key=lambda x: x[1], reverse=True)
                                
                                for feature_name, importance_value in sorted_importance:
                                    importance_data.append({
                                        'feature': feature_name,
                                        'rf_importance': importance_value
                                    })
                                
                                if importance_data:
                                    importance_df = pd.DataFrame(importance_data)
                                    importance_df.to_csv(region_path / f"jja_rf_feature_importance_{var}.csv", index=False)
                                    print(f"  Saved: jja_rf_feature_importance_{var}.csv")
                                    total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving RF feature importance: {e}")

                # 5. Bootstrap Distributions (per variable)
                try:
                    if 'rf_results' in feedback_results:
                        for var, results in feedback_results['rf_results'].items():
                            if 'bootstrap_stats' in results and results['bootstrap_stats']:
                                bs_stats = results['bootstrap_stats']
                                
                                if 'feedback_strength' in bs_stats and 'distribution' in bs_stats['feedback_strength']:
                                    boot_dist = bs_stats['feedback_strength']['distribution']
                                    boot_df = pd.DataFrame({
                                        'bootstrap_sample': range(len(boot_dist)),
                                        'feedback_strength': boot_dist,
                                        'variable': var
                                    })
                                    boot_df.to_csv(region_path / f"jja_bootstrap_distribution_{var}.csv", index=False)
                                    print(f"  Saved: jja_bootstrap_distribution_{var}.csv")
                                    total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving bootstrap distributions: {e}")

                # 6. Analysis Summary Text File
                try:
                    if ('normalization_info' in feedback_results and 
                        feedback_results['normalization_info'] and
                        feedback_results['normalization_info'].get('normalized_strengths')):
                        
                        norm_info = feedback_results['normalization_info']
                        normalized_strengths = norm_info.get('normalized_strengths', {})
                        shap_enabled = feedback_results.get('shap_enabled', False)
                        bootstrap_enabled = feedback_results.get('bootstrap_enabled', False)
                        
                        if normalized_strengths:
                            with open(region_path / "jja_analysis_summary_with_shap.txt", 'w') as f:
                                f.write(f"Enhanced JJA Climate Analysis Summary\n")
                                f.write(f"=" * 60 + "\n\n")
                                f.write(f"Dataset: {feedback_results.get('dataset_name', 'Unknown')}\n")
                                f.write(f"Region: {feedback_results.get('region_name', 'Unknown')}\n")
                                f.write(f"Season: JJA (June-July-August)\n")
                                f.write(f"Total JJA seasons analyzed: {feedback_results.get('total_jja_seasons', 'Unknown')}\n")
                                f.write(f"SHAP Analysis: {'ENABLED' if shap_enabled else 'DISABLED'}\n")
                                f.write(f"Block Bootstrap: {'ENABLED' if bootstrap_enabled else 'DISABLED'}\n\n")
                                
                                f.write(f"Feedback Strength Analysis:\n")
                                f.write(f"Raw JJA feedback strength range: {norm_info.get('raw_min', 0):.6f} to {norm_info.get('raw_max', 0):.6f}\n\n")
                                f.write(f"Normalized JJA feedback strengths (0-1 scale):\n")
                                
                                sorted_vars = sorted(normalized_strengths.items(), key=lambda x: x[1], reverse=True)
                                for var, norm_strength in sorted_vars:
                                    f.write(f"  {var}: {norm_strength:.6f}\n")
                                
                                if bootstrap_enabled and 'rf_results' in feedback_results:
                                    f.write(f"\nBlock Bootstrap Results:\n")
                                    for var in sorted_vars:
                                        var_name = var[0]
                                        if var_name in feedback_results['rf_results']:
                                            res = feedback_results['rf_results'][var_name]
                                            if 'bootstrap_stats' in res and res['bootstrap_stats']:
                                                bs = res['bootstrap_stats']['feedback_strength']
                                                f.write(f"  {var_name}:\n")
                                                f.write(f"    Mean: {bs['mean']:.6f}\n")
                                                f.write(f"    Std: {bs['std']:.6f}\n")
                                                f.write(f"    95% CI: [{bs['ci_lower']:.6f}, {bs['ci_upper']:.6f}]\n")
                                                f.write(f"    Significant: {bs['ci_lower'] > 0}\n")
                                
                                if shap_enabled:
                                    f.write(f"\nSHAP Analysis Notes:\n")
                                    f.write(f"  • SHAP values provide interpretability\n")
                                    f.write(f"  • Variable dependencies show non-linear relationships\n")
                                    f.write(f"  • Directional effects indicate increase/decrease influence\n")
                                    f.write(f"  • SHAP-Granger comparison reveals method agreements\n")
                            
                            print(f"  Saved: jja_analysis_summary_with_shap.txt")
                            total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving analysis summary: {e}")

                # 7. Granger Causality Matrix
                try:
                    if ('gc_matrix' in feedback_results and 
                        feedback_results['gc_matrix'] is not None and
                        not feedback_results['gc_matrix'].empty):
                        
                        gc_matrix = feedback_results['gc_matrix']
                        gc_matrix.to_csv(region_path / "jja_granger_causality_matrix.csv")
                        print(f"  Saved: jja_granger_causality_matrix.csv")
                        total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving GC matrix: {e}")

                # 8. Detailed Granger Causality
                try:
                    if 'gc_detailed' in feedback_results and feedback_results['gc_detailed']:
                        gc_detailed_data = []
                        for link, details in feedback_results['gc_detailed'].items():
                            try:
                                cause, effect = link.split('_causes_')
                                gc_detailed_data.append({
                                    'causality_link': f'{cause}_causes_{effect}',
                                    'primary_p_value_lag2': details.get('primary_p_value', 1.0),
                                    'primary_lag': details.get('primary_lag', 2),
                                    'primary_is_significant': details.get('primary_is_significant', False),
                                    'best_p_value': details.get('min_p_value', 1.0),
                                    'best_lag': details.get('best_lag', 1),
                                    'best_is_significant': details.get('best_is_significant', False)
                                })
                            except ValueError:
                                continue

                        if gc_detailed_data:
                            gc_df = pd.DataFrame(gc_detailed_data)
                            gc_df = gc_df.sort_values('primary_p_value_lag2')
                            gc_df.to_csv(region_path / "jja_granger_causality_detailed.csv", index=False)
                            print(f"  Saved: jja_granger_causality_detailed.csv")
                            total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving detailed GC results: {e}")

                # 9. Processed Seasonal Data
                try:
                    if 'processed_data' in feedback_results:
                        processed_data = feedback_results['processed_data']
                        if isinstance(processed_data, pd.DataFrame) and not processed_data.empty:
                            processed_data.to_csv(region_path / "processed_jja_seasonal_data.csv")
                            print(f"  Saved: processed_jja_seasonal_data.csv")
                            total_files_saved += 1
                except Exception as e:
                    print(f"  Error saving processed data: {e}")

        print(f"\nTotal files saved: {total_files_saved}")
        print(f"Results saved to: {output_path}")
        print(f"SHAP Integration: {'ENABLED' if SHAP_AVAILABLE else 'DISABLED'}")
        
        if hasattr(self, 'n_bootstrap') and self.n_bootstrap > 0:
            print(f"Block Bootstrap: ENABLED ({self.n_bootstrap} samples)")
        else:
            print(f"Block Bootstrap: DISABLED")
        
        return total_files_saved > 0


def setup_jja_analyzer(n_bootstrap=0):
    """Setup JJA analyzer with optional bootstrap"""
    analyzer = ClimateMutualFeedbackAnalyzer_JJA_SHAP(
        max_lags=6,
        significance_level=0.05,
        n_estimators=200,
        random_state=42,
        n_bootstrap=n_bootstrap
    )
    print(f"\n✓ JJA Analyzer ready (bootstrap: {n_bootstrap} samples)")
    return analyzer


def quick_upload_and_process(analyzer, dataset_name, file_paths, start_year=2071, end_year=2100):
    """Upload and process data in one step for JJA analysis"""
    success = analyzer.upload_multiple_files(dataset_name, file_paths)
    if not success:
        return False
    
    success = analyzer.process_to_jja_regional(dataset_name, start_year, end_year)
    if not success:
        return False
    
    print(f"\n✓ {dataset_name} ready for JJA analysis!")
    return True


def run_complete_analysis(analyzer, dataset_name, output_dir=None):
    """Run complete JJA analysis for all regions and save results"""
    all_results = analyzer.run_feedback_analysis_all_regions(dataset_name)
    
    if not all_results:
        return None
    
    for region_name in all_results.keys():
        analyzer.generate_feedback_report(dataset_name, region_name)
    
    if output_dir is None:
        output_dir = f"jja_results_{dataset_name}"
    
    analyzer.save_feedback_results(output_dir)
    print(f"\n✓ JJA results saved to: {output_dir}")
    return all_results


# ============================================================================
# MAIN EXECUTION - YOUR FILE PATHS GO HERE!
# ============================================================================

if __name__ == "__main__":
    print("="*80)
    print("JJA CLIMATE MUTUAL FEEDBACK ANALYSIS")
    print("With SHAP + Block Bootstrap Uncertainty")
    print("="*80)
    
    # ========================================================================
    # SETUP ANALYZER
    # ========================================================================
    # Set n_bootstrap=0 to disable (faster, ~10 min runtime)
    # Set n_bootstrap=1000 to enable (slower, ~30-40 hours runtime)
    
    my_analyzer = setup_jja_analyzer(n_bootstrap=1000)  # Start with 0 for testing!
    
    # ========================================================================
    # YOUR NETCDF FILE PATHS - EDIT THESE! ⬇️⬇️⬇️
    # ========================================================================
    
    cmip6_files = {
        'temperature': '/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/tas/CMCC-ESM2_tas_future126_20712100_seasonal.nc',
        'precipitable_water': '/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/prw/CMCC-ESM2_prw_future126_20712100_seasonal.nc',
        'precipitation': '/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_future126_20712100_seasonal.nc',
        'evaporation': '/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_future126_20712100_seasonal.nc',
        'soil_moisture': '/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrsos/CMCC-ESM2_mrsos_future126_20712100_seasonal.nc',
        'runoff': '/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrro/CMCC-ESM2_mrro_future126_20712100_seasonal.nc'
    }
    
    # ========================================================================
    # UPLOAD AND PROCESS
    # ========================================================================
    
    print("\nUploading and processing JJA data...")
    upload_success = quick_upload_and_process(
        my_analyzer, 
        'cmip6',  # Dataset name
        cmip6_files,  # Your file paths
        start_year=2071,  # First JJA year
        end_year=2100     # Last JJA year
    )
    
    if not upload_success:
        print("✗ Data upload/processing failed. Check your file paths above!")
        exit(1)
    
    # ========================================================================
    # RUN COMPLETE ANALYSIS
    # ========================================================================
    
    print("\nRunning complete JJA mutual feedback analysis...")
    results = run_complete_analysis(
        my_analyzer,
        'cmip6',  # Dataset name
        'cmip6_jja_results_with_bootstrap'  # Output directory name
    )
    
    # ========================================================================
    # DONE!
    # ========================================================================
    
    if results:
        print("\n" + "="*80)
        print("✓ SUCCESS! JJA ANALYSIS COMPLETE")
        print("="*80)
        print(f"\nAnalyzed regions: {list(results.keys())}")
        print(f"Results saved to: cmip6_jja_results_with_bootstrap/")
        print(f"\nFeatures computed:")
        print(f"  • Feedback strength with 95% confidence intervals")
        print(f"  • Statistical significance testing")
        if SHAP_AVAILABLE:
            print(f"  • SHAP variable dependencies")
            print(f"  • SHAP directional effects")
            print(f"  • SHAP-Granger comparison")
        print(f"  • Granger causality relationships")
        print(f"\nCSV files created per region: ~30")
        print(f"Total CSV files: ~180 (6 regions × 30 files)")
        print("="*80)
    else:
        print("\n✗ Analysis failed. Check error messages above.")

# ============================================================================
# END OF SCRIPT
# ============================================================================


✓ SHAP library available - enhanced interpretability enabled!
JJA CLIMATE MUTUAL FEEDBACK ANALYSIS
With SHAP + Block Bootstrap Uncertainty
Enhanced JJA Climate Mutual Feedback Analyzer Initialized!
Supported datasets: CMIP6, ERA5, Generic
Expected variables: ['temperature', 'precipitable_water', 'precipitation', 'evaporation', 'soil_moisture', 'runoff']
Regions to analyze: ['Global', 'Tropics', 'Subtropics_N', 'Subtropics_S', 'Mid_Latitudes_N', 'Mid_Latitudes_S']
Season: JJA (June + July + August)
Expected data period: Dec 2020 - Nov 2050 → JJA seasons 2021-2050 (30 seasons)

Analysis methods:
  • Random Forest feature importance
  • Granger Causality testing
  • SHAP interpretability (ENABLED)
  • Block Bootstrap uncertainty (ENABLED)

Parameters:
  • max_lags: 6 JJA seasons
  • significance_level: 0.05
  • n_estimators: 200
  • random_state: 42
  • n_bootstrap: 1000 samples
  • Block bootstrap: ENABLED (1000 samples)
  • Block length: auto-calculated

✓ JJA Analyzer ready (bootstrap: